# AgriSight: full pipeline on Colab

This notebook runs **every experiment** in the project and writes the results tables, the figure and the README summary.

1. Runtime → Change runtime type → **T4 GPU** → Save
2. Runtime → **Run all**

**Time:** training all models takes several hours, which is more than one free Colab session. That's fine: checkpoints and results are saved to your Google Drive, and finished runs are skipped. If Colab disconnects, open the notebook again and **Run all**. It continues where it stopped.

**Order:** setup → data → smoke test (10 min) → vision experiments → LLM/VLM setup → assistant experiments → report → push results to GitHub.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/AgriSight'
os.makedirs(DRIVE, exist_ok=True)
os.environ['AGRISIGHT_RUNS_DIR'] = f'{DRIVE}/runs'
os.environ['AGRISIGHT_INDEX_DIR'] = f'{DRIVE}/index'
os.environ['EPOCHS'] = '8'          # 8 epochs per model keeps the total time reasonable on a T4
print('Results will be saved to', DRIVE)

In [ ]:
%cd /content
!test -d AgriSight || git clone https://github.com/Rajeevreddy-2006/AgriSight.git
%cd /content/AgriSight
!git pull -q
# Colab already has torch/torchvision; install the rest
!pip install -q timm sentence-transformers rank-bm25 pypdf onnx onnxruntime python-multipart gTTS accelerate
!pip install -q -e . --no-deps
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Data
Downloaded to the fast local disk (about 10 minutes). After a disconnect this re-runs; the manifests point to the same paths, so saved checkpoints stay valid.

In [ ]:
!bash scripts/download_data.sh
!python scripts/prepare_data.py --plantvillage data/raw/PlantVillage-Dataset/raw/color --plantdoc data/raw/PlantDoc-Dataset

## 3. Smoke test
Runs a tiny version of every vision step into `runs_smoke/` to catch problems before the long runs.

In [ ]:
!SMOKE=1 AGRISIGHT_RUNS_DIR=/content/runs_smoke bash scripts/run_all.sh vision

## 4. Vision experiments (RQ1, RQ2, RQ5)
The long part. Trains 6 classifiers + DANN + 2 phone models, evaluates each on lab and field photos, and compresses the phone model. Re-run this cell after a disconnect.

In [ ]:
!bash scripts/run_all.sh vision

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open('docs/RESULTS.md').read().replace('figures/', 'docs/figures/')))
if os.path.exists('docs/figures/lab_vs_field.png'):
    display(Image('docs/figures/lab_vs_field.png', width=800))

## 5. LLM and vision-language model (Ollama)
Installs Ollama in Colab and downloads Qwen2.5 (text) and Qwen2.5-VL (vision). About 10 GB, 5–10 minutes.

In [ ]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull qwen2.5:7b-instruct
!ollama pull qwen2.5vl:7b
!curl -s http://localhost:11434/api/tags | head -c 300

## 6. Assistant experiments (RQ3, RQ4)

Before this step, add **official guideline PDFs** (ICAR, state agricultural universities) to `knowledge_base/` in your GitHub repo and write your question set as `eval/questions.jsonl` (see `eval/questions.sample.jsonl` for the format). Without them the run uses the placeholder documents and 8 sample questions. That is fine for testing, but the numbers won't mean much.

In [ ]:
!bash scripts/run_all.sh assistant
display(Markdown(open('docs/RESULTS.md').read().replace('figures/', 'docs/figures/')))

## 7. Try the assistant

In [ ]:
import csv, sys
sys.path.insert(0, 'src')
from PIL import Image as PILImage
from agrisight.config import load_config
from agrisight.factory import build_agent
cfg = load_config()
best = !python scripts/make_report.py --print-best
agent = build_agent(cfg, ckpt=f"{os.environ['AGRISIGHT_RUNS_DIR']}/{best[-1]}/best.pt")

row = next(r for r in csv.DictReader(open('data/manifests/field_test.csv')) if 'blight' in r['label'])
img = PILImage.open(row['path'])
display(img.resize((300, int(300 * img.height / img.width))))
print('True label:', row['label'])
out = agent.ask('What is this problem and what should I do?', image=img, language='te')
print('Diagnosis:', out['diagnosis']['display'] if out['diagnosis'] else None)
print(out['answer'])
print('Unsupported numbers:', out['unsupported_numbers'])

## 8. Save results to GitHub

Create a token: GitHub → Settings → Developer settings → Fine-grained tokens → access to **AgriSight** only, permission **Contents: Read and write**. Paste it when asked. The token is not saved in the notebook.

In [ ]:
from getpass import getpass
token = getpass('GitHub token: ')
!git config user.name "Rajeevreddy-2006"
!git config user.email "Rajeevreddy-2006@users.noreply.github.com"
!git add docs/RESULTS.md docs/figures README.md
!git commit -m "Add experiment results"
!git push https://{token}@github.com/Rajeevreddy-2006/AgriSight.git HEAD:main
del token